# Phase 2 — Local Live Demo (PCAP → score)

Runs **on your Mac (CPU)** with the trained checkpoint pulled from Drive.

Demo arc for the presentation:
1. Show the model card + thresholds.
2. Replay `data/demo_pcaps/benign_http.pcap` → all flows print BENIGN.
3. Replay `data/demo_pcaps/ssh_bruteforce.pcap` → flows print ATTACK with `SYN Flag Count` / `Flow IAT Min` as the top-attribution features.
4. (Optional) re-run with `--live en0` against your own machine for a live capture.

Make sure you ran `scripts/make_demo_pcaps.py` once to populate `data/demo_pcaps/`.

In [ ]:
from pathlib import Path
import sys, json
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))
CKPT  = REPO / 'models' / 'phase2_local' / 'checkpoints' / 'ft_svdd_final.pt'
PCAPS = REPO / 'data' / 'demo_pcaps'
CKPT.exists(), list(PCAPS.glob('*.pcap'))

## 1. Model card

In [ ]:
from cps_ad.torch_ft_svdd import load_checkpoint
model, extra = load_checkpoint(str(CKPT), map_location='cpu')
thr = json.loads((CKPT.parent / 'thresholds.json').read_text())
print('config :', model.cfg)
print('center norm:', float(model.center.norm()))
print('thresholds :', thr)

## 2. Replay benign PCAP

In [ ]:
!python {REPO / 'scripts' / 'demo_score_packets.py'} --pcap {PCAPS / 'benign_http.pcap'} --ckpt {CKPT} --top-k 3

## 3. Replay SSH bruteforce PCAP

In [ ]:
!python {REPO / 'scripts' / 'demo_score_packets.py'} --pcap {PCAPS / 'ssh_bruteforce.pcap'} --ckpt {CKPT} --top-k 3

## 4. (Optional) live capture

Requires `sudo`. Replace `en0` with your interface (`ifconfig` to list).

In [ ]:
# !sudo -E python {REPO / 'scripts' / 'demo_score_packets.py'} --live en0 --duration 30 --ckpt {CKPT} --quiet-benign